# VGG11 from Scratch on Oxford-IIIT Pet (Colab-Compatible)

This notebook converts the repository training pipeline into an interactive notebook workflow that runs locally or on Google Colab.


In [ ]:

# Colab/repo bootstrap
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/<your-org>/VGG11_with_Codex.git'  # update if needed
REPO_DIR = Path('/content/VGG11_with_Codex') if IN_COLAB else Path.cwd()

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    print(f'Running in Colab. Working directory: {Path.cwd()}')
else:
    print(f'Running locally. Working directory: {Path.cwd()}')


In [ ]:
# Install dependencies
!pip -q install -r requirements.txt


In [ ]:
# Optional: mount Google Drive for dataset/checkpoint persistence
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:

from dataclasses import dataclass
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import transforms

from data.pets_dataset import OxfordIIITPetDataset
from models.classification import VGG11Classifier


In [ ]:

@dataclass
class TrainConfig:
    data_root: str = '/content/oxford-iiit-pet'  # update to your dataset location
    epochs: int = 10
    batch_size: int = 32
    lr: float = 1e-3
    weight_decay: float = 5e-4
    dropout: float = 0.5
    num_workers: int = 2
    val_ratio: float = 0.1
    seed: int = 42
    save_dir: str = 'checkpoints'

cfg = TrainConfig()
cfg


In [ ]:

def accuracy(logits: torch.Tensor, targets: torch.Tensor) -> float:
    preds = torch.argmax(logits, dim=1)
    return (preds == targets).float().mean().item()


def run_epoch(model, loader, criterion, device, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    total_acc = 0.0
    total_samples = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        if is_train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        batch_size = images.size(0)
        total_samples += batch_size
        total_loss += loss.item() * batch_size
        total_acc += accuracy(logits.detach(), labels) * batch_size

    return total_loss / total_samples, total_acc / total_samples


In [ ]:

# Data pipeline

torch.manual_seed(cfg.seed)

train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

full_train = OxfordIIITPetDataset(root=cfg.data_root, split='trainval', transform=train_tfms)
val_size = int(len(full_train) * cfg.val_ratio)
train_size = len(full_train) - val_size

gen = torch.Generator().manual_seed(cfg.seed)
train_subset, val_subset = random_split(full_train, [train_size, val_size], generator=gen)
val_subset.dataset.transform = eval_tfms

test_ds = OxfordIIITPetDataset(root=cfg.data_root, split='test', transform=eval_tfms)

train_loader = DataLoader(train_subset, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers)
val_loader = DataLoader(val_subset, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)

print(f'train: {len(train_subset)} | val: {len(val_subset)} | test: {len(test_ds)}')


In [ ]:

# Model, optimizer, and training loop

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

model = VGG11Classifier(num_classes=37, dropout_p=cfg.dropout).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

best_val_acc = 0.0
save_dir = Path(cfg.save_dir)
save_dir.mkdir(parents=True, exist_ok=True)
best_path = save_dir / 'vgg11_classifier_best.pt'

for epoch in range(1, cfg.epochs + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, device, optimizer=optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, criterion, device, optimizer=None)

    print(
        f"Epoch {epoch:03d}/{cfg.epochs} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(
            {
                'model_state_dict': model.state_dict(),
                'epoch': epoch,
                'val_acc': val_acc,
                'config': cfg.__dict__,
            },
            best_path,
        )
        print(f'Saved new best checkpoint to: {best_path}')


In [ ]:

# Final test evaluation with best checkpoint
ckpt = torch.load(best_path, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
test_loss, test_acc = run_epoch(model, test_loader, criterion, device, optimizer=None)
print(f'Test metrics | loss={test_loss:.4f} acc={test_acc:.4f}')


## Optional: call the original script directly

If you prefer the CLI path inside Colab, run:

```bash
!python train.py --data-root /content/oxford-iiit-pet --epochs 30 --batch-size 32 --lr 1e-3 --dropout 0.5
```
